# RoadVision AI

Your original pasted code, in its original order, divided into notebook cells. Only section headings and cell boundaries have been added.

The original `/content` paths, fixed dataset counts, cleaning operations, and training commands are preserved. This notebook has not been executed.


## 1. Imports


In [ ]:
from pathlib import Path
from collections import Counter
import shutil
import random

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import yaml




## 2. Download the dataset


In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download(
    "ashfakyeafi/road-vehicle-images-dataset"
)

print("Dataset downloaded to:", dataset_path)



## 3. Inspect the dataset folder


In [ ]:
import os

print("Dataset folder:")
print(dataset_path)

print("\nFiles and folders:")
for item in os.listdir(dataset_path):
    print(item)



## 4. Count images and annotations


In [ ]:
from pathlib import Path

dataset = Path(dataset_path)

images = [
    p for p in dataset.rglob("*")
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
]

labels = list(dataset.rglob("*.txt"))
yaml_files = list(dataset.rglob("*.yaml"))

print("Images:", len(images))
print("Annotations:", len(labels))
print("YAML files:", len(yaml_files))



## 5. Preview the first image


In [ ]:
if images:
    image = Image.open(images[0])

    plt.figure(figsize=(8, 6))
    plt.imshow(image)
    plt.axis("off")
    plt.title("First image from our dataset")
    plt.show()
else:
    print("No images found!")



## 6. Show the dataset tree


In [ ]:
def show_tree(folder, max_depth=3):
    folder = Path(folder)

    for path in sorted(folder.rglob("*")):
        depth = len(path.relative_to(folder).parts)

        if depth <= max_depth:
            indent = "  " * (depth - 1)
            print(indent + path.name)

show_tree(dataset, max_depth=2)



## 7. Load the dataset configuration


In [ ]:
yaml_files = sorted(dataset.rglob("data_1.yaml"))

if not yaml_files:
    raise FileNotFoundError(
        "No data.yaml found. Check the dataset folder."
    )

print("Found configuration files:")
for file in yaml_files:
    print(file)

YAML_PATH = yaml_files[0]

with open(YAML_PATH, "r") as f:
    config = yaml.safe_load(f)

print("\nDataset configuration:")
print(config)



## 8. Read class names


In [ ]:
names = config["names"]

if isinstance(names, dict):
    class_names = {
        int(k): str(v) for k, v in names.items()
    }
else:
    class_names = {
        i: str(name) for i, name in enumerate(names)
    }

num_classes = len(class_names)

print("\nNumber of classes:", num_classes)
display("Classes:", class_names)

assert set(class_names) == set(range(num_classes)), \
    "Class IDs must be consecutive, starting at 0."



## 9. Collect image files


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

image_files = sorted([
    p for p in dataset.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
])

print("Total images:", len(image_files))

print("\nFirst five images:")
for image in image_files[:5]:
    print(image.relative_to(dataset))



## 10. Match images with labels


In [ ]:
def find_label(image_path):
    parts = list(image_path.parts)

    image_positions = [
        i for i, part in enumerate(parts)
        if part.lower() == "images"
    ]

    if not image_positions:
        raise ValueError(
            f"Cannot locate 'images' folder for {image_path}"
        )

    # Replace the images directory with labels.
    idx = image_positions[-1]
    parts[idx] = "labels"

    return Path(*parts).with_suffix(".txt")


label_paths = [find_label(p) for p in image_files]

print("Images:", len(image_files))
print(
    "Existing matching labels:",
    sum(p.exists() for p in label_paths)
)
print(
    "Images missing labels:",
    sum(not p.exists() for p in label_paths)
)



## 11. Inspect an annotation


In [ ]:
existing_labels = [
    p for p in label_paths if p.is_file()
]

if existing_labels:
    example_label = existing_labels[0]

    print("Label file:", example_label.name)
    print("\nContents:")
    print(example_label.read_text())
else:
    print("No annotation files were found.")



## 12. Inspect annotation size and contents


In [ ]:
example_label = existing_labels[0]

print("File:", example_label.name)
print("File size:", example_label.stat().st_size, "bytes")
print("Content:", repr(example_label.read_text()))



## 13. Find non-empty annotations


In [ ]:
non_empty_labels = [
    p for p in existing_labels
    if p.read_text().strip()
]

print("Total label files:", len(existing_labels))
print("Non-empty labels:", len(non_empty_labels))
print(
    "Empty labels:",
    len(existing_labels) - len(non_empty_labels)
)

if non_empty_labels:
    example_label = non_empty_labels[0]

    print("\nLabel file:", example_label.name)
    print("\nContents:")
    print(example_label.read_text())
else:
    print("All annotation files are empty!")



## 14. List empty annotations


In [ ]:
empty_labels = [
    p for p in existing_labels
    if not p.read_text().strip()
]

print("Empty annotation files:")

for label in empty_labels:
    print(label)



## 15. Find images for empty annotations


In [ ]:
from pathlib import Path

dataset = Path(dataset_path)

for label_path in empty_labels:
    print("\nLooking for:", label_path.name)

    image_name = label_path.stem

    matches = [
        p for p in dataset.rglob("*")
        if p.is_file()
        and p.stem == image_name
        and p.suffix.lower() in {
            ".jpg", ".jpeg", ".png", ".webp", ".bmp"
        }
    ]

    if matches:
        for image in matches:
            print("✅ Image found:", image)
    else:
        print("❌ No matching image found")



## 16. Preview empty-label images in grayscale


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

for label_path in empty_labels:

    matches = [
        p for p in dataset.rglob("*")
        if p.is_file()
        and p.stem == label_path.stem
        and p.suffix.lower() in {
            ".jpg", ".jpeg", ".png", ".webp", ".bmp"
        }
    ]

    for image_path in matches:
        with Image.open(image_path) as img:
            plt.figure(figsize=(10, 6))
            plt.imshow(img.convert("L"))

        plt.title(image_path.name)
        plt.axis("off")
        plt.show()



## 17. Preview empty-label images in color


In [ ]:
for label_path in empty_labels:

    image_base = Path(
        *[
            "images" if part == "labels" else part
            for part in label_path.parts
        ]
    ).with_suffix("")

    found = False

    for extension in [".jpg", ".jpeg", ".png", ".webp"]:
        image_path = image_base.with_suffix(extension)

        if image_path.exists():
            image = Image.open(image_path)

            plt.figure(figsize=(9, 5))
            plt.imshow(image)
            plt.title(f"Empty annotation: {label_path.name}")
            plt.axis("off")
            plt.show()

            found = True
            break

    if not found:
        print("Image not found for:", label_path.name)



## 18. Check unmatched filenames


In [ ]:
image_stems = {p.stem for p in images}
label_stems = {p.stem for p in existing_labels}

print("Images with no matching label:",
      len(image_stems - label_stems))

print("Labels with no matching image:",
      len(label_stems - image_stems))

print("\nExample unmatched labels:")
print(list(label_stems - image_stems)[:10])



## 19. Copy the dataset for cleaning


In [ ]:
clean_dataset = Path("/content/road_vehicle_clean")

shutil.copytree(
    dataset,
    clean_dataset,
    dirs_exist_ok=True
)

print("Clean dataset created:", clean_dataset)



## 20. Remove empty-label pairs from the copy


In [ ]:
for label_path in empty_labels:

    clean_label = clean_dataset / label_path.relative_to(dataset)

    matching_images = [
        p for p in image_files
        if p.stem == label_path.stem
        and p.parent.parent.name == label_path.parent.parent.name
    ]

    if len(matching_images) == 1:
        clean_image = (
            clean_dataset / matching_images[0].relative_to(dataset)
        )

        clean_image.unlink()
        clean_label.unlink()

        print("Removed:", clean_image.name)
        print("Removed:", clean_label.name)

    else:
        print("Skipped:", label_path.name)



## 21. Count remaining files


In [ ]:
clean_images = [
    p for p in clean_dataset.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
]

clean_labels = list(clean_dataset.rglob("*.txt"))

print("Images remaining:", len(clean_images))
print("Labels remaining:", len(clean_labels))



## 22. Validate class IDs


In [ ]:
invalid_classes = []

for label_path in clean_labels:

    for line_number, line in enumerate(
        label_path.read_text().splitlines(), start=1
    ):
        values = line.split()

        if not values:
            continue

        try:
            class_id = int(values[0])

            if class_id not in class_names:
                invalid_classes.append(
                    (label_path.name, line_number, class_id)
                )

        except ValueError:
            invalid_classes.append(
                (label_path.name, line_number, values[0])
            )

print("Invalid class IDs:", len(invalid_classes))

for error in invalid_classes[:10]:
    print(error)



## 23. Validate bounding boxes


In [ ]:
import math

invalid_boxes = []

for label_path in clean_labels:

    for line_number, line in enumerate(
        label_path.read_text().splitlines(), start=1
    ):
        values = line.split()

        try:
            if len(values) != 5:
                raise ValueError("Expected 5 values")

            class_value, x, y, w, h = map(float, values)

            if not all(math.isfinite(v) for v in
                       [class_value, x, y, w, h]):
                raise ValueError("Non-finite number")

            if not class_value.is_integer():
                raise ValueError("Invalid class ID format")

            if not (0 <= x <= 1 and 0 <= y <= 1):
                raise ValueError("Invalid center coordinates")

            if not (0 < w <= 1 and 0 < h <= 1):
                raise ValueError("Invalid box size")

            if (
                x - w/2 < -1e-6 or
                x + w/2 > 1 + 1e-6 or
                y - h/2 < -1e-6 or
                y + h/2 > 1 + 1e-6
            ):
                raise ValueError("Box outside image")

        except ValueError as error:
            invalid_boxes.append(
                (label_path.name, line_number, str(error))
            )

print("Invalid annotation lines:", len(invalid_boxes))

for error in invalid_boxes[:len(invalid_boxes)]:
    print(error)



## 24. Summarize invalid annotation files


In [ ]:
affected_files = Counter(
    filename for filename, line, error in invalid_boxes
)

print("Invalid annotation lines:", len(invalid_boxes))
print("Affected files:", len(affected_files))

for filename, count in affected_files.items():
    print(f"{filename}: {count} invalid lines")



## 25. Remove pairs with invalid boxes


In [ ]:
deleted = []

for image_path in clean_images:

    label_path = find_label(image_path)

    if label_path.name in affected_files:
        image_path.unlink()
        label_path.unlink()

        deleted.append(image_path.name)
        print("Deleted:", image_path.name)

print("\nTotal images deleted:", len(deleted))



## 26. Check remaining annotations


In [ ]:
clean_images = [p for p in clean_images if p.exists()]
clean_labels = [p for p in clean_labels if p.exists()]

print("Remaining images:", len(clean_images))
print("Remaining labels:", len(clean_labels))

remaining_errors = []

for filename, line_number, error in invalid_boxes:
    if any(p.name == filename for p in clean_labels):
        remaining_errors.append((filename, line_number, error))

print("Remaining previously detected errors:", len(remaining_errors))



## 27. Check image integrity


In [ ]:
corrupted_images = []

for image_path in clean_images:
    try:
        with Image.open(image_path) as img:
            img.verify()

    except Exception as error:
        corrupted_images.append((image_path, str(error)))

print("Total images checked:", len(clean_images))
print("Corrupted images:", len(corrupted_images))

for image_path, error in corrupted_images:
    print(image_path.name, ":", error)



## 28. Check image decoding


In [ ]:
unreadable_images = []

for image_path in clean_images:
    try:
        with Image.open(image_path) as img:
            img.load()

    except Exception as error:
        unreadable_images.append((image_path, str(error)))

print("Images that cannot be decoded:", len(unreadable_images))



## 29. Count objects per class


In [ ]:
class_counts = Counter()

for label_path in clean_labels:
    with open(label_path, "r") as f:
        for line in f:
            if line.strip():
                class_id = int(line.split()[0])
                class_counts[class_id] += 1

print("Class Distribution:\n")

for class_id, class_name in class_names.items():
    print(f"{class_name}: {class_counts[class_id]}")



## 30. Build the class distribution table


In [ ]:
class_distribution = pd.DataFrame([
    {
        "Class ID": class_id,
        "Class Name": class_name,
        "Count": class_counts[class_id]
    }
    for class_id, class_name in class_names.items()
])

class_distribution = class_distribution.sort_values(
    by="Count", ascending=False
)

display(class_distribution)



## 31. Plot class distribution


In [ ]:
plt.figure(figsize=(12, 6))

plt.bar(
    class_distribution["Class Name"],
    class_distribution["Count"],
    color="steelblue"
)

plt.title("RoadVision AI - Class Distribution")
plt.xlabel("Vehicle Classes")
plt.ylabel("Number of Objects")

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()



## 32. Collect training and validation images


In [ ]:
train_images = [
    p for p in clean_images
    if p.parent.parent.name == "train"
]

valid_images = [
    p for p in clean_images
    if p.parent.parent.name == "valid"
]

print("Train candidates:", len(train_images))
print("Validation:", len(valid_images))

assert len(train_images) == 2702
assert len(valid_images) == 298



## 33. Create the test split


In [ ]:
from sklearn.model_selection import train_test_split

train_images, test_images = train_test_split(
    sorted(train_images),
    test_size=405,
    random_state=42
)

print("Train:", len(train_images))
print("Validation:", len(valid_images))
print("Test:", len(test_images))



## 34. Create split directories


In [ ]:
SPLIT_ROOT = Path("/content/road_vehicle_split")

for split in ["train", "valid", "test"]:
    (SPLIT_ROOT / "images" / split).mkdir(
        parents=True, exist_ok=True
    )
    (SPLIT_ROOT / "labels" / split).mkdir(
        parents=True, exist_ok=True
    )

print("Dataset folders created!")



## 35. Copy image-label pairs into splits


In [ ]:
splits = {
    "train": train_images,
    "valid": valid_images,
    "test": test_images
}

for split_name, image_list in splits.items():

    for image_path in image_list:
        label_path = find_label(image_path)

        if not label_path.is_file():
            raise FileNotFoundError(label_path)

        shutil.copy2(
            image_path,
            SPLIT_ROOT / "images" / split_name / image_path.name
        )

        shutil.copy2(
            label_path,
            SPLIT_ROOT / "labels" / split_name / label_path.name
        )

    print(split_name, ":", len(image_list))



## 36. Verify split counts


In [ ]:
for split in ["train", "valid", "test"]:

    image_count = len(list(
        (SPLIT_ROOT / "images" / split).iterdir()
    ))

    label_count = len(list(
        (SPLIT_ROOT / "labels" / split).glob("*.txt")
    ))

    print(
        f"{split}: {image_count} images, "
        f"{label_count} labels"
    )

    assert image_count == label_count

assert len(train_images) == 2297
assert len(valid_images) == 298
assert len(test_images) == 405

print("Dataset split verified!")



## 37. Clone YOLOv9


In [ ]:
%cd /content/yolov9

!git clone https://github.com/WongKinYiu/yolov9.git



## 38. Check YOLOv9 files


In [ ]:
from pathlib import Path

repo = Path("/content/yolov9")

files_to_check = [
    "train_dual.py",
    "models/detect/yolov9-c.yaml",
    "data/hyps/hyp.scratch-high.yaml",
    "requirements.txt"
]

for file in files_to_check:
    print(file, ":", (repo / file).exists())





## 39. Install YOLOv9 dependencies


In [ ]:
!pip install -r requirements.txt



## 40. Check GPU availability


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))



## 41. Write the training configuration


In [ ]:
from pathlib import Path
import yaml

data_config = {
    "train": str(SPLIT_ROOT / "images/train"),
    "val": str(SPLIT_ROOT / "images/valid"),
    "test": str(SPLIT_ROOT / "images/test"),
    "nc": len(class_names),
    "names": [class_names[i] for i in range(len(class_names))]
}

DATA_YAML = SPLIT_ROOT / "data.yaml"

with open(DATA_YAML, "w") as f:
    yaml.safe_dump(data_config, f, sort_keys=False)

print(DATA_YAML.read_text())



## 42. Download pretrained weights


In [ ]:
%cd /content/yolov9

!wget -nc \
  https://github.com/WongKinYiu/yolov9/releases/download/v0.1/yolov9-c.pt \
  -O yolov9-c.pt



## 43. Check weights and runtime


In [ ]:
import torch

weights_path = Path("/content/yolov9/yolov9-c.pt")

print("Weights exist:", weights_path.is_file())

if weights_path.is_file():
    print("Weights size (MB):",
          round(weights_path.stat().st_size / 1024**2, 2))

print("Dataset config:", DATA_YAML.is_file())
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))



## 44. Disable Weights & Biases


In [ ]:
import os

os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_DISABLED"] = "true"




## 45. Train YOLOv9


In [ ]:
%cd /content/yolov9

!python train_dual.py \
    --weights /content/yolov9/yolov9-c.pt \
    --cfg models/detect/yolov9-c.yaml \
    --data {DATA_YAML} \
    --hyp data/hyps/hyp.scratch-high.yaml \
    --epochs 10 \
    --batch-size 4 \
    --img 640 \
    --device 0 \
    --workers 2 \
    --name roadvision_yolov9c



## 46. Show training results


In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

result_dir = Path("/content/yolov9/runs/train/roadvision_yolov9c")

if (result_dir / "results.png").exists():
    img = Image.open(result_dir / "results.png")

    plt.figure(figsize=(16, 10))
    plt.imshow(img)
    plt.axis("off")
    plt.show()
else:
    print("Training results not found yet.")



## 47. Find result plots


In [ ]:
from pathlib import Path

runs_dir = Path("/content/yolov9/runs/train")

result_images = list(runs_dir.rglob("results.png"))

print("Results images found:", len(result_images))

for path in result_images:
    print(path)



## 48. Generate plots from result CSV files


In [ ]:
%cd /content/yolov9

from pathlib import Path
from utils.plots import plot_results

csv_files = list(
    Path("/content/yolov9/runs/train").rglob("results.csv")
)

print("CSV files found:", len(csv_files))

for csv_file in csv_files:
    print("Processing:", csv_file)

    plot_results(file=csv_file)